In [ ]:
# HyperOpt 패키지 설치 명령어 (주석 처리됨, 필요 시 #을 제거하고 실행)
# HyperOpt는 베이지안 최적화(Bayesian Optimization) 기반의 하이퍼 파라미터 튜닝 라이브러리로,
# 그리드 서치/랜덤 서치보다 효율적으로 최적 하이퍼 파라미터를 탐색할 수 있음.
#pip install hyperopt

In [ ]:
# hyperopt 라이브러리를 임포트
import hyperopt

# 설치된 hyperopt 버전을 출력 (버전에 따라 API 차이가 있을 수 있으므로 확인 용도)
print(hyperopt.__version__)

In [ ]:
# HyperOpt의 hp 모듈 임포트 (하이퍼 파라미터 검색 공간 정의에 사용)
from hyperopt import hp

# -10 ~ 10까지 1간격을 가지는 입력 변수 x와 -15 ~ 15까지 1간격으로 입력 변수 y 설정.
# hp.quniform(label, low, high, q): low ~ high 범위에서 q 간격의 균일 분포로 샘플링
#   - 정수형 또는 일정 간격을 가지는 변수를 탐색할 때 사용 (반환값은 실수형이므로 필요시 int()로 변환)
# 검색 공간(search space)은 dict 형태로, key는 변수명, value는 hp 함수의 분포 객체
search_space = {'x': hp.quniform('x', -10, 10, 1), 'y': hp.quniform('y', -15, 15, 1) }

In [ ]:
# STATUS_OK: 목적 함수가 정상적으로 수행되었음을 나타내는 상수 (HyperOpt 내부에서 사용)
from hyperopt import STATUS_OK

# 목적 함수를 생성. 변숫값과 변수 검색 공간을 가지는 딕셔너리를 인자로 받고, 특정 값을 반환
# - HyperOpt는 이 목적 함수의 반환값(loss)을 '최소화'하는 방향으로 탐색
# - 따라서 최댓값을 찾고 싶다면 부호를 반대로 바꾸어 반환해야 함
def objective_func(search_space):
    # search_space 딕셔너리에서 x, y 값 추출
    x = search_space['x']
    y = search_space['y']
    # 예시 목적 함수: x^2 - 20y (최소화 대상)
    retval = x**2 - 20*y
    
    # 단순한 예제이므로 반환값(loss)만 반환 (status 없이도 동작)
    return retval

In [ ]:
# HyperOpt의 주요 함수/객체 임포트
# - fmin: 목적 함수를 최소화하는 최적 입력값을 찾아주는 함수
# - tpe: Tree-structured Parzen Estimator. 베이지안 최적화의 핵심 알고리즘
# - Trials: 매 시도(iteration)마다의 입력값과 결괏값을 저장하는 객체
from hyperopt import fmin, tpe, Trials
import numpy as np

# 입력 결괏값을 저장한 Trials 객체값 생성.
# 학습 진행 과정의 입력값, 출력값, 상태 등을 기록 → 후속 분석에 활용 가능
trial_val = Trials()

# 목적 함수의 최솟값을 반환하는 최적 입력 변숫값을 5번의 입력값 시도(max_evals=5)로 찾아냄.
# fn: 최소화할 목적 함수
# space: 탐색할 검색 공간(dict)
# algo: 최적화 알고리즘 (tpe.suggest = 베이지안 최적화 TPE 방식)
# max_evals: 목적 함수 평가 횟수 (= 시도 횟수)
# trials: 시도 기록을 저장할 Trials 객체
# rstate: 결과 재현을 위한 난수 시드 (numpy의 default_rng 사용)
best_01 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=5
               , trials=trial_val, rstate=np.random.default_rng(seed=0))
# 5회 시도를 통해 찾은 최적의 (x, y) 출력
print('best:', best_01)

In [ ]:
# 새로운 Trials 객체 생성 (이전 시도 기록과 분리)
trial_val = Trials()

# max_evals를 20회로 늘려서 재테스트
# 시도 횟수를 늘리면 더 다양한 후보를 탐색하여 더 좋은 최적값을 찾을 가능성이 높아짐
# (단, 시간/비용도 함께 증가)
best_02 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=20
               , trials=trial_val, rstate=np.random.default_rng(seed=0))
# 20회 시도를 통해 찾은 최적의 (x, y) 출력 (5회 결과보다 더 나은 값 기대)
print('best:', best_02)

In [ ]:
# fmin( )에 인자로 들어가는 Trials 객체의 result 속성에 파이썬 리스트로 목적 함수 반환값들이 저장됨
# 리스트 내부의 개별 원소는 {'loss':함수 반환값, 'status':반환 상태값} 와 같은 딕셔너리임. 
# 시도 횟수만큼 results 리스트가 생성되며, 각 시도의 결과(loss)를 확인 가능
print(trial_val.results)

In [ ]:
# Trials 객체의 vals 속성에 {'입력변수명':개별 수행 시마다 입력된 값 리스트} 형태로 저장됨.
# 각 시도(iteration)마다 어떤 x, y 값이 입력으로 사용됐는지 확인할 수 있음
# 예: {'x': [1.0, -3.0, ...], 'y': [10.0, 5.0, ...]}
print(trial_val.vals)

In [ ]:
# 시도 결과를 표 형태로 확인하기 위해 pandas 임포트
import pandas as pd

# results에서 loss 키값에 해당하는 밸류들을 추출하여 list로 생성. 
# 리스트 컴프리헨션으로 각 시도(loss_dict)의 'loss' 값만 추출
losses = [loss_dict['loss'] for loss_dict in trial_val.results]

# DataFrame으로 생성.
# x, y 입력값과 그에 대응하는 손실(loss) 값을 한 표에 정리 → 어떤 입력에서 최소 손실이 나왔는지 확인하기 쉬움
result_df = pd.DataFrame({'x': trial_val.vals['x'], 'y': trial_val.vals['y'], 'losses': losses})
result_df

### HyperOpt를 이용한 XGBoost 하이퍼 파라미터 최적화

In [ ]:
# 아래 코드는 이전에 수록된 코드라 책에는 싣지 않았습니다. 
# 위스콘신 유방암 데이터셋을 로드하고 학습/테스트 분리를 위한 준비 작업
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
# 코드 실행 시 발생하는 경고(warning) 메시지를 무시하도록 설정
warnings.filterwarnings('ignore')

# 위스콘신 유방암 데이터셋 로드
dataset = load_breast_cancer()

# 피처 데이터를 DataFrame으로 변환 후 'target' 컬럼 추가
cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
cancer_df['target']= dataset.target
# 피처(독립 변수)와 레이블(종속 변수) 분리
X_features = cancer_df.iloc[:, :-1]  # 마지막 컬럼('target') 제외
y_label = cancer_df.iloc[:, -1]      # 마지막 컬럼('target')만

In [ ]:
# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
# random_state=156: 동일한 분할 결과 재현을 위한 시드 고정값
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156 )

# 앞에서 추출한 학습 데이터를 다시 학습과 검증 데이터로 분리
# 검증(validation) 데이터는 최적 하이퍼 파라미터 적용 모델의 조기 중단 평가에 사용
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )

In [ ]:
# XGBoost 하이퍼 파라미터 검색 공간을 정의하기 위한 hp 모듈 임포트
from hyperopt import hp

# max_depth는 5에서 20까지 1간격으로, min_child_weight는 1에서 2까지 1간격으로
# colsample_bytree는 0.5에서 1사이, learning_rate는 0.01에서 0.2 사이 정규 분포된 값으로 검색.
# - hp.quniform(label, low, high, q): 정수형 파라미터 탐색 (균등 분포 + 일정 간격 q)
# - hp.uniform(label, low, high): 실수형 파라미터 탐색 (균등 분포)
# 주의: hp.quniform 반환값도 실수형이므로 XGBClassifier에 넘길 때 int()로 변환 필요
xgb_search_space = {'max_depth': hp.quniform('max_depth', 5, 20, 1),                  # 트리 최대 깊이
                    'min_child_weight': hp.quniform('min_child_weight', 1, 2, 1),     # 리프 노드 최소 가중치 합
                    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),          # 학습률
                    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),       # 트리 생성 시 사용할 컬럼 비율
                   }

In [ ]:
# 교차 검증(cross-validation) 기반 모델 평가를 위한 함수 임포트
from sklearn.model_selection import cross_val_score
# XGBoost의 사이킷런 래퍼 분류 클래스 임포트
from xgboost import XGBClassifier
# 목적 함수가 정상 수행됐음을 나타내는 상태 상수
from hyperopt import STATUS_OK

# fmin()에서 입력된 search_space 값으로 입력된 모든 값은 실수형임.
# XGBClassifier의 정수형 하이퍼 파라미터는 정수형 변환을 해줘야 함.
# 정확도는 높을수록 더 좋은 수치임. -1 * 정확도를 곱해서 큰 정확도 값일수록 최소가 되도록 변환
# (HyperOpt fmin은 손실 함수를 '최소화'하므로, 정확도와 같이 '최대화'할 지표는 부호를 반대로)
def objective_func(search_space):
    # 수행 시간 절약을 위해 nestimators는 100으로 축소
    # 정수형 파라미터(max_depth, min_child_weight)는 반드시 int()로 캐스팅
    xgb_clf = XGBClassifier(n_estimators=100, max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            learning_rate=search_space['learning_rate'],
                            colsample_bytree=search_space['colsample_bytree'],
                            eval_metric='logloss')   # 평가 지표는 logloss (XGBoost 1.3+ 경고 회피)
    # 3-fold 교차 검증으로 정확도(accuracy) 측정 → 길이 3인 배열 반환
    accuracy = cross_val_score(xgb_clf, X_train, y_train, scoring='accuracy', cv=3)
    
    # accuracy는 cv=3 개수만큼 roc-auc 결과를 리스트로 가짐. 이를 평균해서 반환하되 -1을 곱함.
    # HyperOpt 목적 함수 반환 형식: {'loss': 최소화할 값, 'status': STATUS_OK}
    return {'loss':-1 * np.mean(accuracy), 'status': STATUS_OK}


In [ ]:
# HyperOpt의 fmin, tpe, Trials 임포트
from hyperopt import fmin, tpe, Trials

# 시도 기록 저장용 Trials 객체 생성
trial_val = Trials()
# 베이지안 최적화 실행 → XGBoost 모델의 최적 하이퍼 파라미터 탐색
# fn: 위에서 정의한 objective_func (loss 최소화 함수)
# space: xgb_search_space (XGBoost 하이퍼 파라미터 탐색 공간)
# algo: tpe.suggest (TPE 알고리즘 기반 베이지안 최적화)
# max_evals=50: 최대 50회 시도하며 최적값 탐색
# trials: 각 시도 결과를 기록할 Trials 객체
# rstate: 재현성을 위한 numpy 난수 시드(seed=9)
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=50, # 최대 반복 횟수를 지정합니다.
            trials=trial_val, rstate=np.random.default_rng(seed=9))
# 50회 시도 중 가장 좋은 손실(가장 높은 정확도)을 보인 하이퍼 파라미터 조합 출력
print('best:', best)


In [ ]:
# 베이지안 최적화로 찾은 최적 하이퍼 파라미터를 보기 좋게 출력
# - 실수형 파라미터(colsample_bytree, learning_rate)는 소수점 5자리까지 반올림
# - 정수형 파라미터(max_depth, min_child_weight)는 int()로 캐스팅
print('colsample_bytree:{0}, learning_rate:{1}, max_depth:{2}, min_child_weight:{3}'.format(
    round(best['colsample_bytree'], 5), round(best['learning_rate'], 5),
    int(best['max_depth']), int(best['min_child_weight'])))

In [ ]:
# 분류 성능 평가에 사용할 사이킷런 메트릭 함수들 임포트
from sklearn.metrics import confusion_matrix, accuracy_score   # 오차행렬, 정확도
from sklearn.metrics import precision_score, recall_score      # 정밀도, 재현율
from sklearn.metrics import f1_score, roc_auc_score            # F1 스코어, ROC-AUC

# 분류 모델 성능 평가 결과를 한 번에 출력해주는 사용자 정의 함수
# y_test: 실제 정답 레이블
# pred: 예측 클래스 (0/1)
# pred_proba: 양성 클래스 예측 확률 (ROC-AUC 계산에 필요)
def get_clf_eval(y_test, pred=None, pred_proba=None):
    # 오차행렬: [[TN, FP], [FN, TP]] 형태로 반환
    confusion = confusion_matrix( y_test, pred)
    # 정확도: 전체 중 올바르게 예측한 비율
    accuracy = accuracy_score(y_test , pred)
    # 정밀도: 양성으로 예측한 것 중 실제 양성 비율 (TP / (TP+FP))
    precision = precision_score(y_test , pred)
    # 재현율: 실제 양성 중 양성으로 예측한 비율 (TP / (TP+FN))
    recall = recall_score(y_test , pred)
    # F1 스코어: 정밀도와 재현율의 조화 평균
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가 
    # ROC-AUC: 임계값 변화에 따른 분류 성능을 종합적으로 나타내는 지표 (1에 가까울수록 우수)
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    # 평가 지표 5종을 소수점 4자리까지 한 줄로 출력
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
# 베이지안 최적화로 찾은 최적 하이퍼 파라미터를 적용한 XGBClassifier 생성
# n_estimators=400: 본 학습 단계에서는 트리 수를 충분히 늘려 조기 중단으로 최적 시점 결정
# learning_rate, max_depth, min_child_weight, colsample_bytree: HyperOpt가 찾은 최적값을 사용
xgb_wrapper = XGBClassifier(n_estimators=400,
                            learning_rate=round(best['learning_rate'], 5),
                            max_depth=int(best['max_depth']),
                            min_child_weight=int(best['min_child_weight']),
                            colsample_bytree=round(best['colsample_bytree'], 5)
                           )

# 조기 중단을 위한 평가 데이터셋 리스트 (학습용, 검증용 모두 포함)
evals = [(X_tr, y_tr), (X_val, y_val)]
# 학습 수행
# early_stopping_rounds=50: 검증 데이터의 logloss가 50회 연속 개선되지 않으면 학습 조기 종료
# eval_metric='logloss': 평가 지표
# eval_set: 학습 중 평가에 사용할 데이터셋
# verbose=True: 라운드별 학습 진행 상황 출력
xgb_wrapper.fit(X_tr, y_tr, early_stopping_rounds=50, eval_metric='logloss',
                eval_set=evals, verbose=True)

# 테스트 데이터에 대한 예측 클래스(0/1) 산출
preds = xgb_wrapper.predict(X_test)
# 테스트 데이터에 대한 양성(1) 클래스 예측 확률만 슬라이싱 (ROC-AUC 계산용)
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

# 최적 하이퍼 파라미터를 적용한 XGBoost 모델의 최종 성능 평가
get_clf_eval(y_test, preds, pred_proba)
